# KSW Estimator Tester

This notebook tests the KSW estimator pipeline using `Initializor` and `Generator` classes.

In [ ]:
# Setup logging FIRST to prevent Jupyter double-printing
import logging
import os

# Suppress TensorFlow and CUDA warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # 0=all, 1=info, 2=warning, 3=error
os.environ["CUDA_VISIBLE_DEVICES"] = ""  # Disable GPU

# Configure logging to suppress verbose external packages
logging.getLogger("tensorflow").setLevel(logging.ERROR)
logging.getLogger("tensorrt").setLevel(logging.ERROR)

from mlpng.utils import setup_logging

logger = setup_logging(__name__, level=logging.INFO, base_level=logging.WARNING)

## Configuration

Edit these settings to customize the test run.

In [ ]:
# Configuration - edit these as needed
CONFIG = {
    "settings_file": "settings/n64.json",
    "nsims": 100,
    "shapes": ["local", "equilateral", "orthogonal"],
    "lensing_options": [False, True],  # Both unlensed and lensed
    "force_generation": True,
    "fnl_min": -1000.0,
    "fnl_max": 1000.0,
    "n_sigma": 5,  # Number of sigma levels to report
}

## Initialize Generator

Set up the `Generator` with cosmology and transfer functions.

In [ ]:
import numpy as np
from mlpng.generator import Generator
from mlpng.initializor import Initializor
from mlpng.utils import plot_predictions

# Build command-line style args for Initializor (generates and saves KSW)
# Use 'all' if multiple shapes, otherwise use the single shape name
if len(CONFIG["shapes"]) > 1:
    shapes_arg = "all"
else:
    shapes_arg = CONFIG["shapes"][0]

init_argv = [
    CONFIG["settings_file"],
    "--shapes",
    shapes_arg,
    "--nsims",
    str(CONFIG["nsims"]),
    "--force_generation",  # Always retrain KSW for clean test
    "--lensing" if True in CONFIG["lensing_options"] else "--no-lensing",
]

logger.info("Initializing KSW estimators with Initializor...")
logger.info("Init args: %s", init_argv)
initializer = Initializor(argv=init_argv, log_level=logging.INFO)
initializer.run()
logger.info("KSW estimators saved to: %s", initializer.mc_file)

# Now initialize Generator to load and use KSW estimators
gen_argv = [
    CONFIG["settings_file"],
    "--nsims",
    str(CONFIG["nsims"]),
    "--shapes",
    shapes_arg,
    "--force_generation" if CONFIG["force_generation"] else "--no-force_generation",
    "--lensing" if True in CONFIG["lensing_options"] else "--no-lensing",
]

gen = Generator(argv=gen_argv)
logger.info(
    "Generator initialized: lmax=%d, nside=%d, nsims=%d", gen.lmax, gen.nside, gen.nsims
)

## Helper Functions

Define reusable functions for KSW initialization, estimation, and analysis.

In [ ]:
def print_sigma_stats(residuals, sigma, n_sigma=5):
    """Print confidence interval statistics."""
    expected = {1: 68.27, 2: 95.45, 3: 99.73, 4: 99.994, 5: 99.99994}

    print(f"\n{'='*60}")
    print(f"Confidence Interval Statistics (σ = {sigma:.4f})")
    print(f"{'='*60}")
    print(f"{'Level':<10} {'Count':<15} {'Percentage':<15} {'Expected':<15}")
    print(f"{'-'*60}")

    for i in range(1, n_sigma + 1):
        within = np.sum(np.abs(residuals) < i * sigma)
        pct = 100 * within / len(residuals)
        exp = expected.get(i, 100.0)
        print(
            f"{i}σ{'':<8} {within}/{len(residuals):<12} {pct:>6.2f}%{'':<8} {exp:>6.2f}%"
        )

    print(f"{'='*60}")
    print(f"Mean residual: {np.mean(residuals):.4f}")
    print(f"Std residual:  {np.std(residuals):.4f}")

## Generate Gaussian ALMs

Generate base Gaussian alms (used for all shape/lensing combinations).

In [ ]:
# Generate Gaussian alms using Generator (unlensed base)
logger.info("Generating %d Gaussian alms using Generator...", CONFIG["nsims"])
alm_l = gen.generate_alm()

# Generate random true fnl values (same for all tests)
rng = np.random.default_rng(42)  # Fixed seed for reproducibility
fnl_true = rng.uniform(CONFIG["fnl_min"], CONFIG["fnl_max"], (CONFIG["nsims"], 1, 1))

logger.info("Base ALM data generated: shape=%s", alm_l.shape)

## Run All Shape/Lensing Combinations

Loop through all shapes and lensing options, running KSW estimation for each.

In [ ]:
from tqdm.auto import tqdm

# Store results for summary
results = {}
pol_idxs = gen.pol_idxs()

for lensed in CONFIG["lensing_options"]:
    lensed_str = "lensed" if lensed else "unlensed"

    for shape_str in tqdm(CONFIG["shapes"], desc=f"{lensed_str}"):
        label = f"{shape_str} ({lensed_str})"
        print(f"\n{'#'*70}")
        print(f"## {label.upper()}")
        print(f"{'#'*70}")

        # 1. Load pre-trained KSW estimator using Generator.get_ksw
        logger.info("Loading KSW estimator for %s...", label)
        ksw = gen.get_ksw(shape_str, lensed=lensed)

        # 2. Generate non-Gaussian alms using Generator methods
        logger.info("Generating non-Gaussian alms for %s...", shape_str)
        if shape_str == "local":
            alm_nl = gen.generate_alm_nl(alm_l)
        else:
            alm_nl = gen.generate_alm_nl_shape(alm_l, shape_str, ksw)

        # 3. Lens if needed using Generator.lens_alms
        if lensed:
            alm_l_use, alm_phi = gen.lens_alms(alm_l)
            alm_nl_use, _ = gen.lens_alms(alm_nl, alm_phi)
        else:
            alm_l_use = alm_l[:, pol_idxs]
            alm_nl_use = alm_nl

        # 4. Combine alms and run estimation
        alm_combined = alm_l_use + fnl_true * alm_nl_use
        print(f"alm_combined shape: {alm_combined.shape}")

        fisher = ksw.compute_fisher()
        sigma = 1.0 / np.sqrt(fisher)
        logger.info("Fisher: %.4f, σ: %.4f", fisher, sigma)

        estimates, _, _, _ = ksw.compute_estimate_batch(
            lambda idx: gen.icov_func(alm_combined[idx], lensed=lensed),
            range(CONFIG["nsims"]),
            fisher=fisher,
        )
        estimates = estimates.T.flatten()
        fnl_flat = fnl_true.flatten()

        # 5. Compute residuals and print statistics
        residuals = estimates - fnl_flat
        print_sigma_stats(residuals, sigma, CONFIG["n_sigma"])

        # 6. Plot predictions
        plot_predictions(
            fnl_flat,
            estimates,
            sigma=sigma,
            title=f"KSW Estimator: {label}",
            show_n_sigma=CONFIG["n_sigma"],
            save=False,
            show=True,
            close=True,
        )

        # Store results
        results[label] = {
            "fisher": fisher,
            "sigma": sigma,
            "mean_residual": np.mean(residuals),
            "std_residual": np.std(residuals),
        }

logger.info("All combinations complete!")

## Summary

Display a summary table of all results.

In [ ]:
import pandas as pd

# Create summary DataFrame
summary_data = []
for label, res in results.items():
    summary_data.append(
        {
            "Configuration": label,
            "Fisher": f"{res['fisher']:.4f}",
            "σ": f"{res['sigma']:.4f}",
            "Mean Residual": f"{res['mean_residual']:.4f}",
            "Std Residual": f"{res['std_residual']:.4f}",
        }
    )

df = pd.DataFrame(summary_data)
print("\n" + "=" * 80)
print("SUMMARY OF ALL CONFIGURATIONS")
print("=" * 80)
display(df)